# 4. MESA Deploy and Runner

## Pre-requisites

- MESA Deploy credentials
- A GPU with at least 24GB of RAM

## Overview

Once a MESA model is trained, it needs to be run alongside the real data to perform the actual standardisation. `vLLM` is a powerful inference engine for LLMs, which can be used either directly as a library (_offline_), or run behind an HTTP server to service requests (_remote_). vLLM also has a relatively complex configuration, which depends on the GPU being used for inference. Deploy exists to simplify the use of vLLM, providing a clear interface for running in either offline or remote mode, and providing a set of configurations for common GPUs.

Deploy also automates the secure acquisition of weights from the bucket we pushed our weights to in the previous notebook, hence the need for additional credentials. It can also pull weights from a local source, and cache remote weights to that same local source to avoid re-download overhead.

<p align="center">
  <img src="https://londonaicentre.github.io/MESA-Deploy/local/architecture.png" width="700" alt="Overview of the MESA deploy architecture">
</p>

With Deploy providing inference, a gap exists for the task of gathering data from a source, efficiently batching it, sending it to the server, tracking progress and formatting outputs. This is filled by Runner, which pairs with Deploy to provide the complete inference pipeline.

⭐ Standardising whole documents at scale, rather than extracting individual entities for individual tasks, is another key contribution of MESA.

In this notebook we will use Deploy -- specifically its `Local` tool, with it also having a feature for deploying models to the cloud (AWS) -- to download the model we trained previously, run it and observe the standardisation process in practice.

> 📓 As we have only generated a small number of samples, it is unlikely that our model will give us suitable results in respect of our KRAS mutation problem; in practice, we would generate thousands of samples to ensure a high-performing model. As such, our aim here is just to illustrate the general use of these tools, and use existing outputs to understand how we solve our problem. 

## Steps

1. To our `.env` file where Deploy credentials should already live, we will need to add the name of the model we finetuned in the previous notebook, in the form `<model_name>_<major version>_<minor version>_<patch version>` (e.g. `MODEL=oncotest_0_1_0`). We could also add information here on the type of GPU we are using, and Deploy would select a vLLM configuration for us. We could also control aspects of the configuration directly. However, we will stick with the default vLLM configuration for now.

2. Unlike the packages we've used previously, both Deploy (Local) and Runner are CLI tools. We therefore don't usually import them into python, but instead run them directly via the CLI. In the case of Local, we will need this to be running constantly in the background in server (remote) mode (above). To do this, we _will_ use python, but just as a mechanism for launching a process that uses the CLI tool. When you are ready, run the code below to start the server. You'll get lots of output as the model is processed, which may be of interest.

>❗Depending on the base model used, the weights downloaded by the below command can be several GB, which can take both network time and bandwidth.

>❗Once the weights are downloaded, they will be loaded into your GPU which will increase its load and temperature, and may affect the general operation of your machine and other processes.

In [ ]:
import os
import subprocess
import sys
import threading

In [ ]:
# much of the below is boilerplate to enable us to run the server from this notebook. In reality, all we need is `mesalocal -v`!
proc = subprocess.Popen(
    ["mesalocal", "-v"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ | {"PYTHONPATH": os.getcwd()},
)
assert proc.stdout is not None
threading.Thread(target=sys.stdout.writelines, args=(proc.stdout,), daemon=True).start()

3. Now our server is running in the background with our `oncotest` model, we can pass it some data to standardise. To do this, we can use the [example free-text](../data/sample/oncoqwen_sample_data_input.json) we've already seen.

4. We can now invoke Runner as an additional tool to ingest this data, queue it up and pass it to Local, receive the results, check they are valid as per the schema, and output the formatted results to a folder. Runner uses the configuration defined in [`config.yml`](config.yml) which, like Docsynth, controls its operation. Take a moment to inspect the config, and then run the below command to start inference. As before, this will produce lots of output, some of which may be of interest.

In [ ]:
# again, much of the below is boilerplate to enable us to run Runner from this notebook. In reality, all we need is `mesa_runner --config config.yml`!
os.environ.pop("NAME", None)
mesa_runner = subprocess.Popen(
    ["mesa_runner", "--config", "config.yml"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
assert mesa_runner.stdout is not None
threading.Thread(
    target=sys.stdout.writelines, args=(mesa_runner.stdout,), daemon=True
).start()

5. The final output command from above should let you know how many documents were valid (i.e. how many were correct under the schema, meaning the model did its job correctly). Quite a few will probably fail! Examine the outputs in `data/output` and see what the model got wrong, and may have got right. Compare this to the valid outputs from a fully fine-tuned cancer model in [`example outputs`](../data/sample/oncoqwen_sample_data_output.jsonl) that are a companion to the example free-text earlier.

> 📓 The example outputs illustrate how MESA models solve the KRAS problem we posed earlier: looking at the outputs we can see that for each of the notes where KRAS is mentioned (but none of the respective patients have the mutation), the corresponding schema lists `{\"biomarker\":\"kras\",\"biomarker_name_desc\":\"KRAS\",\"biomarker_status\":\"negative\" .. }`. In other words, correctly appreciates the negation context, such that a query against the schema would not only yield a count (of 0), but a reliable one.

6. We can now move to the [final notebook](../validate/validate.ipynb).